# Fase 4: Sinais Alternativos e Processamento de Linguagem Natural (FinBERT)

Este notebook demonstra o fluxo de inferência com notícias financeiras e cálculo da divergência estatística preço-sentimento:
1. **Extração de Sentimento via FinBERT** ($S_{\text{news}} = P(\text{positivo}) - P(\text{negativo})$).
2. **Padronização Temporal ($Z$-Score Móvel a 5 dias)** de retornos de preço e sentimento diário.
3. **Índice de Divergência Preço-Sentimento ($D_t = Z(S_t) - Z(R_t)$)** para identificação de assimetrias e alertas de mercado.

In [1]:
# Cell 1: Importações e Simulação de Notícias
import os
import sys
import pandas as pd
import numpy as np

# Adicionar diretório raiz ao path
sys.path.insert(0, os.path.abspath('..'))

from src.features.sentiment import FinBERTSentimentAnalyzer, compute_sentiment_divergence

# Instanciação do analisador FinBERT (fallback automático se modelo remoto/GPU não estiver disponível)
analyzer = FinBERTSentimentAnalyzer()

# Notícias simuladas por ativo
news_data = {
    'AAPL': [
        "Apple reports record quarterly revenue and iPhone sales.",
        "Supply chain disruptions might impact upcoming iPad delivery schedules.",
        "Analyst upgrades Apple rating to Strong Buy following AI announcements."
    ],
    'MSFT': [
        "Cloud growth slows down amid increased competition in the enterprise sector.",
        "Microsoft faces regulatory scrutiny over latest acquisition deal.",
        "Quarterly earnings hit expectation, but guidance remains weak."
    ],
    'GOOGL': [
        "Alphabet faces new antitrust investigation from European regulators.",
        "Ad revenue drops for second consecutive quarter.",
        "Google unveils groundbreaking quantum computing research paper."
    ]
}

In [2]:
# Cell 2: Inferência de Sentimento via FinBERT
sentiment_results = {}
for ticker, headlines in news_data.items():
    scores = analyzer.predict_headlines(headlines)
    sentiment_results[ticker] = np.mean(scores) # Score médio do dia

print("=== SCORES DE SENTIMENTO EXTRAÍDOS VIA FINBERT ===")
for ticker, score in sentiment_results.items():
    print(f"Ativo: {ticker} | Score de Sentimento Diário: {score:.4f}")

In [3]:
# Cell 3: Cálculo de Divergência Simulada
dates = pd.date_range(start="2024-01-01", periods=10, freq="B")
df_sim_prices = pd.DataFrame({
    'AAPL': 150 + np.cumsum([0.5, 0.2, -0.1, 0.4, 0.8, 1.2, 0.1, -0.3, 0.5, 0.9]),
    'MSFT': 300 + np.cumsum([-1.0, -0.5, -2.0, -1.5, -0.8, -1.2, -0.5, -1.0, -0.2, -0.4]),
    'GOOGL': 120 + np.cumsum([1.5, 2.0, 1.8, 2.5, 3.0, 1.2, 0.8, 2.1, 1.5, 1.9])
}, index=dates)

# Simulação de sentimento diário acumulado ao longo do tempo
df_sim_sentiment = pd.DataFrame({
    'AAPL': [0.2, 0.3, 0.1, 0.4, 0.5, 0.7, 0.8, 0.6, 0.7, 0.85],
    'MSFT': [-0.1, -0.2, -0.5, -0.4, -0.6, -0.7, -0.8, -0.6, -0.7, -0.72],
    'GOOGL': [-0.2, -0.4, -0.5, -0.3, -0.6, -0.7, -0.8, -0.5, -0.6, -0.60]
}, index=dates)

df_divergence = compute_sentiment_divergence(df_sim_prices, df_sim_sentiment, window=5)

print("\n=== ÍNDICE DE DIVERGÊNCIA PREÇO-SENTIMENTO (ÚLTIMA DATA) ===")
print(df_divergence.iloc[-1])